In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch.nn as nn

# 设置中文显示
plt.rcParams["font.family"] = ["Heiti TC"]
plt.rcParams['axes.unicode_minus'] = False

# 生成示例正弦波数据
#x = np.linspace(0, 100, 1000)
#y = np.sin(x) + np.random.normal(0, 0.1, 1000)
# y = pd.read_excel("df_T1.xlsx").WindSpeed.values
# chunks = np.array_split(y, 4)
# y1,y2,y3,y4 = chunks[0],chunks[1],chunks[2],chunks[3]
# y = y4

yy = pd.read_excel("df_Aventa.xlsx").dropna().WindSpeed.values
y=yy[-2500:]

In [ ]:

# 数据归一化
scaler = MinMaxScaler(feature_range=(-1, 1))
y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

class TimeSeriesSlidingWindow(Dataset):
    def __init__(self, data, window_size, step=1, horizon=1):
        self.data = data
        self.window_size = window_size
        self.step = step
        self.horizon = horizon
        self.X, self.y = self._create_windows()
    
    def _create_windows(self):
        X, y = [], []
        max_i = len(self.data) - self.window_size - self.horizon + 1
        for i in range(0, max_i, self.step):
            X.append(self.data[i:i+self.window_size])
            y.append(self.data[i+self.window_size+self.horizon-1])
        return np.array(X), np.array(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.X[idx], dtype=torch.float32).unsqueeze(-1)
        y_tensor = torch.tensor(self.y[idx], dtype=torch.float32)
        return x_tensor, y_tensor

# 参数设置
window_size = 20
step = 1
horizon = 1

# 创建数据集
dataset = TimeSeriesSlidingWindow(y_scaled, window_size, step, horizon)
print(f"总样本数: {len(dataset)}")

# 按时序切分数据集
train_size = int(0.8 * len(dataset))
train_dataset = torch.utils.data.Subset(dataset, range(0, train_size))
test_dataset = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))

print(f"训练集样本数: {len(train_dataset)}")
print(f"测试集样本数: {len(test_dataset)}")

# 创建数据加载器
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

class AdaptivePhysicsInformedLoss(nn.Module):
    def __init__(self, alpha=0.5, lambda_physics=0.1, rho_air=1.225, dt=1.0, 
                 diffusion_coef=0.01, noise_intensity=0.05, 
                 adaptation_rate=0.001, min_lambda=0.01, max_lambda=1.0):
        """
        自适应随机物理信息增强的混合损失函数
        
        Args:
            adaptation_rate: 超参数调整速率
            min_lambda, max_lambda: lambda_physics的最小和最大值
        """
        super(AdaptivePhysicsInformedLoss, self).__init__()
        
        # 将超参数定义为可训练参数（使用对数空间确保正值）
        self.log_lambda_physics = nn.Parameter(torch.log(torch.tensor(lambda_physics)))
        self.log_diffusion_coef = nn.Parameter(torch.log(torch.tensor(diffusion_coef)))
        self.log_noise_intensity = nn.Parameter(torch.log(torch.tensor(noise_intensity)))
        
        # 基础参数
        self.alpha = alpha
        self.beta = 1 - alpha
        self.rho_air = rho_air
        self.dt = dt
        self.adaptation_rate = adaptation_rate
        self.min_lambda = min_lambda
        self.max_lambda = max_lambda
        
        # 损失历史记录
        self.data_loss_history = []
        self.physics_loss_history = []
        self.adaptation_window = 10  # 适应窗口大小
        
    @property
    def lambda_physics(self):
        """获取经过约束的lambda_physics值"""
        return torch.exp(self.log_lambda_physics).clamp(self.min_lambda, self.max_lambda)
    
    @property
    def diffusion_coef(self):
        """获取扩散系数（确保正值）"""
        return torch.exp(self.log_diffusion_coef).clamp(1e-6, 1.0)
    
    @property
    def noise_intensity(self):
        """获取噪声强度（确保正值）"""
        return torch.exp(self.log_noise_intensity).clamp(1e-6, 0.1)

    def forward(self, predictions, targets, input_sequences, device):
        """
        前向传播，包含自适应超参数调整
        """
        # 1. 数据精度损失 - 使用分位数损失 (0.1, 0.5, 0.9)
        quantiles = [0.1, 0.5, 0.9]
        quantile_losses = []
        
        # 确保predictions有三个输出（三个分位数）
        assert predictions.shape[1] == len(quantiles), "预测值应该有3个输出（0.1, 0.5, 0.9分位数）"
        
        for i, q in enumerate(quantiles):
            # 提取当前分位数的预测值
            q_pred = predictions[:, i].unsqueeze(1)  # 保持维度为 [batch_size, 1]
            errors = targets - q_pred
            loss_q = torch.max((q - 1) * errors, q * errors)
            quantile_losses.append(torch.mean(loss_q))
        
        # 计算加权平均的分位数损失
        weights = torch.tensor([0.2, 0.6, 0.2]).to(device)  # 给0.5分位数更高权重
        data_loss = torch.sum(torch.stack(quantile_losses) * weights)
        
        # 2. 随机物理约束损失 - 使用0.5分位数预测数据
        median_predictions = predictions[:, 1].unsqueeze(1)  # 取第二个输出作为0.5分位数
        physics_loss = self._compute_stochastic_physics_constraint(
            median_predictions, targets, input_sequences, device
        )
        
        # 3. 组合损失
        total_loss = data_loss + self.lambda_physics * physics_loss
        
        # 4. 自适应调整超参数（不计算梯度）
        with torch.no_grad():
            self._adapt_hyperparameters(data_loss, physics_loss)
        
        return total_loss, data_loss, physics_loss    
    
    def _adapt_hyperparameters(self, data_loss, physics_loss):
        """根据损失反馈自动调整超参数[6](@ref)"""
        # 记录损失历史
        self.data_loss_history.append(data_loss.item())
        self.physics_loss_history.append(physics_loss.item())
        
        # 保持历史记录长度
        if len(self.data_loss_history) > self.adaptation_window:
            self.data_loss_history.pop(0)
            self.physics_loss_history.pop(0)
        
        # 当有足够历史数据时进行调整
        if len(self.data_loss_history) >= self.adaptation_window:
            # 计算最近的平均损失
            avg_data_loss = np.mean(self.data_loss_history[-5:])
            avg_physics_loss = np.mean(self.physics_loss_history[-5:])
            
            # 调整策略：平衡数据损失和物理损失
            loss_ratio = avg_data_loss / (avg_physics_loss + 1e-8)
            
            # 调整lambda_physics：如果数据损失占主导，增大lambda加强物理约束
            if loss_ratio > 2.0:  # 数据损失远大于物理损失
                adjustment = self.adaptation_rate
            elif loss_ratio < 0.5:  # 物理损失远大于数据损失
                adjustment = -self.adaptation_rate
            else:  # 损失相对平衡
                adjustment = 0.0
            
            # 应用调整（在对数空间）
            new_log_lambda = self.log_lambda_physics.item() + adjustment
            self.log_lambda_physics.copy_(torch.tensor(new_log_lambda))
            
            # 基于训练稳定性调整扩散系数
            if len(self.data_loss_history) >= 10:
                loss_std = np.std(self.data_loss_history[-10:])
                if loss_std < avg_data_loss * 0.1:  # 训练稳定，可以增加物理复杂性
                    self.log_diffusion_coef += self.adaptation_rate * 0.1
                else:  # 训练不稳定，简化物理模型
                    self.log_diffusion_coef -= self.adaptation_rate * 0.1
    
    def _compute_stochastic_physics_constraint(self, predictions, targets, input_sequences, device):
        """计算随机物理约束损失（保持原有实现）"""
        batch_size = predictions.size(0)
        physics_loss = torch.tensor(0.0).to(device)
        
        # 原有的各种物理约束计算
        stochastic_continuity_loss = self._stochastic_mass_conservation(input_sequences, predictions, device)
        stochastic_momentum_loss = self._stochastic_momentum_conservation(input_sequences, predictions, device)
        stochastic_energy_loss = self._stochastic_energy_conservation(input_sequences, predictions, targets, device)
        fokker_planck_loss = self._fokker_planck_constraint(input_sequences, predictions, device)
        
        physics_loss += (stochastic_continuity_loss + stochastic_momentum_loss + 
                        stochastic_energy_loss + fokker_planck_loss)
        
        return physics_loss / batch_size
    
    def _stochastic_mass_conservation(self, input_sequences, predictions, device):
        """
        随机质量守恒约束: ∂ρ/∂t + ∇·(ρu) = η(x,t)
        添加随机项模拟质量源汇的随机波动[1](@ref)
        """
        seq_len = input_sequences.size(1)
        
        if seq_len > 1:
            wind_speeds = input_sequences.squeeze(-1)  # [batch_size, seq_len]
            
            # 计算时间导数 (确定性部分)
            time_gradients = []
            for i in range(seq_len - 1):
                grad = (wind_speeds[:, i+1] - wind_speeds[:, i]) / self.dt
                time_gradients.append(grad)
            
            if time_gradients:
                avg_time_gradient = torch.stack(time_gradients).mean()
                
                # 添加随机项: 空间白噪声 η(x,t)
                batch_size, seq_len_minus_1 = wind_speeds[:, :-1].shape
                spatial_noise = self.noise_intensity * torch.randn(batch_size, seq_len_minus_1).to(device)
                
                # 随机连续性方程残差: ∂ρ/∂t + ∇·(ρu) - η(x,t)
                stochastic_residual = avg_time_gradient + spatial_noise.mean()
                continuity_loss = torch.mean(stochastic_residual ** 2)
                return continuity_loss
        
        return torch.tensor(0.0).to(device)
    
    def _stochastic_momentum_conservation(self, input_sequences, predictions, device):
        """
        随机动量守恒约束: ∂u/∂t + u·∇u = -1/ρ ∇p + ν∇²u + σ dW/dt
        添加随机力项模拟湍流等随机效应[1,4](@ref)
        """
        wind_speeds = input_sequences.squeeze(-1)  # [batch_size, seq_len]
        
        if wind_speeds.size(1) > 2:
            # 确定性部分: 对流项 u·∇u
            convective_term = torch.zeros_like(wind_speeds[:, 0])
            
            for i in range(1, wind_speeds.size(1) - 1):
                grad_u = (wind_speeds[:, i+1] - wind_speeds[:, i-1]) / (2 * self.dt)
                convective_term += wind_speeds[:, i] * grad_u
            
            # 扩散项: ν∇²u (确定性扩散)
            diffusion_term = self._compute_diffusion_term(wind_speeds, device)
            
            # 随机项: σ dW/dt (维纳过程增量)[4](@ref)
            # 模拟随机力，如湍流起伏
            wiener_increment = self.noise_intensity * torch.randn_like(convective_term) * np.sqrt(self.dt)
            
            # 时间导数项
            time_derivative = (wind_speeds[:, -1] - wind_speeds[:, -2]) / self.dt
            
            # 随机动量方程残差
            stochastic_momentum_residual = (time_derivative + convective_term - 
                                          diffusion_term - wiener_increment)
            
            momentum_loss = torch.mean(stochastic_momentum_residual ** 2)
            return momentum_loss
        
        return torch.tensor(0.0).to(device)
    
    def _compute_diffusion_term(self, wind_speeds, device):
        """计算扩散项 ν∇²u 的近似[1](@ref)"""
        batch_size, seq_len = wind_speeds.shape
        
        if seq_len < 3:
            return torch.zeros(batch_size).to(device)
        
        # 使用中心差分计算二阶空间导数
        diffusion_terms = []
        for i in range(1, seq_len - 1):
            laplacian = (wind_speeds[:, i+1] - 2 * wind_speeds[:, i] + wind_speeds[:, i-1]) / (self.dt ** 2)
            diffusion_terms.append(self.diffusion_coef * laplacian)
        
        if diffusion_terms:
            return torch.stack(diffusion_terms).mean(dim=0)
        return torch.zeros(batch_size).to(device)
    
    def _stochastic_energy_conservation(self, input_sequences, predictions, targets, device):
        """
        随机能量守恒约束: ∂E/∂t + ∇·(Eu) = D∇²E + ξ(x,t)
        考虑能量扩散和随机源汇[1](@ref)
        """
        # 动能计算: E = 0.5 * ρ * u²
        predicted_kinetic = 0.5 * self.rho_air * (predictions ** 2)
        target_kinetic = 0.5 * self.rho_air * (targets ** 2)
        
        # 确定性能量变化
        energy_conservation = torch.mean((predicted_kinetic - target_kinetic) ** 2)
        
        # 随机能量源汇项 ξ(x,t)
        energy_sequences = 0.5 * self.rho_air * (input_sequences ** 2)
        energy_noise = self.noise_intensity * torch.randn_like(energy_sequences) * np.sqrt(self.dt)
        
        # 能量扩散项
        energy_diffusion = self._compute_energy_diffusion(energy_sequences, device)
        
        # 随机能量方程残差
        stochastic_energy_residual = energy_conservation + energy_diffusion + energy_noise.mean()
        
        return torch.mean(stochastic_energy_residual ** 2)
    
    def _compute_energy_diffusion(self, energy_sequences, device):
        """计算能量扩散项 D∇²E[1](@ref)"""
        energy_values = energy_sequences.squeeze(-1)
        batch_size, seq_len = energy_values.shape
        
        if seq_len < 3:
            return torch.zeros(batch_size).to(device)
        
        diffusion_terms = []
        for i in range(1, seq_len - 1):
            laplacian_energy = (energy_values[:, i+1] - 2 * energy_values[:, i] + 
                               energy_values[:, i-1]) / (self.dt ** 2)
            diffusion_terms.append(self.diffusion_coef * laplacian_energy)
        
        if diffusion_terms:
            return torch.stack(diffusion_terms).mean(dim=0)
        return torch.zeros(batch_size).to(device)
    
    def _fokker_planck_constraint(self, input_sequences, predictions, device):
        """
        Fokker-Planck方程约束: ∂p/∂t = -∇·[f p] + ½∇²[D p]
        描述概率密度随时间的演化[4](@ref)
        """
        # 将风速序列视为随机过程的实现
        wind_speeds = input_sequences.squeeze(-1)
        batch_size, seq_len = wind_speeds.shape
        
        if seq_len < 2:
            return torch.tensor(0.0).to(device)
        
        # 估计概率密度函数 (简化为高斯假设)
        mean_speeds = torch.mean(wind_speeds, dim=1)
        var_speeds = torch.var(wind_speeds, dim=1)
        
        # Fokker-Planck方程残差计算 (简化版)
        # ∂p/∂t ≈ Δp/Δt
        if seq_len > 1:
            p_start = self._gaussian_pdf(wind_speeds[:, 0], mean_speeds, var_speeds)
            p_end = self._gaussian_pdf(wind_speeds[:, -1], mean_speeds, var_speeds)
            time_derivative_p = (p_end - p_start) / (seq_len * self.dt)
            
            # 漂移项: -∇·[f p] (f为漂移函数，此处简化为平均风速)
            drift_term = -mean_speeds * (p_end - p_start) / (self.dt)
            
            # 扩散项: ½∇²[D p]
            diffusion_term_p = 0.5 * self.diffusion_coef * (p_end - 2 * p_start + p_start) / (self.dt ** 2)
            
            # Fokker-Planck方程残差
            fp_residual = time_derivative_p + drift_term - diffusion_term_p
            fp_loss = torch.mean(fp_residual ** 2)
            
            return fp_loss
        
        return torch.tensor(0.0).to(device)
    
    def _gaussian_pdf(self, x, mean, variance):
        """计算高斯概率密度函数"""
        return torch.exp(-0.5 * (x - mean) ** 2 / variance) / torch.sqrt(2 * np.pi * variance)

# 改进的随机物理感知GRU模型
class StochasticPhysicsInformedGRU(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, dropout=0.2,
                 physics_dim=32):
        """
        随机物理感知GRU模型
        
        Args:
            physics_dim: 物理特征维度，用于编码随机物理过程
        """
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.physics_dim = physics_dim
        
        # 主GRU层
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # 随机物理特征编码器
        self.physics_encoder = nn.Sequential(
            nn.Linear(hidden_dim + input_dim, physics_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(physics_dim, physics_dim)
        )
        
        # 随机噪声注入层 (模拟随机过程)
        self.noise_injection = nn.Sequential(
            nn.Linear(physics_dim + 1, physics_dim),  # +1 for random noise input
            nn.Tanh()
        )
        
        # 最终预测层 (结合确定性预测和随机物理特征)
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim + physics_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 3)
        )
        
    def forward(self, x, hidden=None):
        if hidden is None:
            h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        else:
            h0 = hidden
        
        # GRU确定性特征提取
        gru_out, hidden = self.gru(x, h0)
        last_hidden = gru_out[:, -1, :]  # 最后一个时间步
        
        # 物理特征编码 (结合输入序列的统计特性)
        sequence_stats = torch.std(x, dim=1)  # 序列变异性作为物理特征
        physics_input = torch.cat([last_hidden, sequence_stats], dim=1)
        physics_features = self.physics_encoder(physics_input)
        
        # 随机噪声注入 (模拟随机物理过程)
        batch_size = x.size(0)
        random_noise = torch.randn(batch_size, 1).to(x.device)  # 随机噪声
        physics_with_noise = torch.cat([physics_features, random_noise], dim=1)
        stochastic_physics_features = self.noise_injection(physics_with_noise)
        
        # 结合确定性和随机特征进行预测
        combined_features = torch.cat([last_hidden, stochastic_physics_features], dim=1)
        output = self.output_layer(combined_features)
        
        return output, hidden

def create_adaptive_optimizer(model, loss_module, base_lr=0.001):
    """
    创建分别优化模型参数和超参数的优化器[8](@ref)
    """
    # 模型参数
    model_params = [param for name, param in model.named_parameters() 
                   if 'log_' not in name]
    
    # 超参数（使用较小的学习率）
    hyperparams = [param for name, param in loss_module.named_parameters() 
                  if 'log_' in name]
    
    optimizer = torch.optim.Adam([
        {'params': model_params, 'lr': base_lr},
        {'params': hyperparams, 'lr': base_lr * 0.01}  # 超参数学习率更小
    ], weight_decay=1e-5)
    
    return optimizer

def train_adaptive_physics_modelall(train_loader, test_loader):
    """增强的训练循环，支持超参数自适应"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 初始化自适应物理感知模型
    model = StochasticPhysicsInformedGRU(
        input_dim=1, hidden_dim=128, num_layers=2, dropout=0.3, physics_dim=64
    ).to(device)
    
    # 使用自适应损失函数
    criterion = AdaptivePhysicsInformedLoss(
        alpha=0.6,
        lambda_physics=0.2,
        rho_air=1.225,
        dt=1.0,
        diffusion_coef=0.01,
        noise_intensity=0.05,
        adaptation_rate=0.001,
        min_lambda=0.01,
        max_lambda=1.0
    ).to(device)
    
    # 创建自适应优化器
    optimizer = create_adaptive_optimizer(model, criterion, base_lr=0.001)
    
    # 学习率调度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    # 训练参数
    num_epochs = 100
    best_loss = float('inf')
    patience = 15
    counter = 0
    
    # 存储训练历史和超参数演化
    train_losses = []
    val_losses = []
    data_losses = []
    physics_losses = []
    lambda_history = []
    diffusion_history = []
    noise_history = []
    
    print("开始自适应物理信息训练...")
    
    for epoch in range(num_epochs):
        model.train()
        epoch_total_loss = 0.0
        epoch_data_loss = 0.0
        epoch_physics_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs, _ = model(batch_X)
            
            # 计算自适应损失（内部会自动调整超参数）
            total_loss, data_loss, physics_loss = criterion(
                outputs, batch_y, batch_X, device
            )
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_total_loss += total_loss.item()
            epoch_data_loss += data_loss.item()
            epoch_physics_loss += physics_loss.item()
        
        # 验证集评估
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device).unsqueeze(1)
                outputs, _ = model(batch_X)
                val_loss_item, _, _ = criterion(outputs, batch_y, batch_X, device)
                val_loss += val_loss_item.item()
        
        # 计算平均损失
        epoch_total_loss /= len(train_loader)
        epoch_data_loss /= len(train_loader)
        epoch_physics_loss /= len(train_loader)
        val_loss /= len(test_loader)
        
        # 记录当前超参数值
        lambda_history.append(criterion.lambda_physics.item())
        diffusion_history.append(criterion.diffusion_coef.item())
        noise_history.append(criterion.noise_intensity.item())
        
        train_losses.append(epoch_total_loss)
        val_losses.append(val_loss)
        data_losses.append(epoch_data_loss)
        physics_losses.append(epoch_physics_loss)
        
        # 定期打印训练状态和超参数值
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}')
            print(f'总损失: {epoch_total_loss:.6f} | 数据损失: {epoch_data_loss:.6f} | '
                  f'物理损失: {epoch_physics_loss:.6f} | 验证损失: {val_loss:.6f}')
            print(f'超参数 - λ_physics: {criterion.lambda_physics.item():.6f}, '
                  f'扩散系数: {criterion.diffusion_coef.item():.6f}, '
                  f'噪声强度: {criterion.noise_intensity.item():.6f}')
        
        # 学习率调度
        scheduler.step(val_loss)
        
        # 早停机制
        if val_loss < best_loss:
            best_loss = val_loss
            counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'criterion_state_dict': criterion.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_loss': best_loss,
                'epoch': epoch,
                'hyperparameter_history': {
                    'lambda': lambda_history,
                    'diffusion': diffusion_history,
                    'noise': noise_history
                }
            }, 'best_adaptive_physics_modelall_fen.pth')
        else:
            counter += 1
            if counter >= patience:
                print(f"早停触发于第 {epoch+1} 轮")
                break
    
    return model, criterion, {
        'lambda_history': lambda_history,
        'diffusion_history': diffusion_history,
        'noise_history': noise_history,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'data_losses': data_losses,      # 新增
        'physics_losses': physics_losses  # 新增
    }

def visualize_hyperparameter_evolution(training_history):
    """可视化超参数在训练过程中的演化"""
    plt.figure(figsize=(15, 10))
    
    # 1. 超参数演化
    plt.subplot(2, 2, 1)
    plt.plot(training_history['lambda_history'], label='λ_physics', linewidth=2)
    plt.plot(training_history['diffusion_history'], label='扩散系数', linewidth=2)
    plt.plot(training_history['noise_history'], label='噪声强度', linewidth=2)
    plt.title('超参数自适应演化')
    plt.xlabel('训练轮次')
    plt.ylabel('超参数值')
    plt.legend()
    plt.grid(True)
    
    # 2. 损失曲线
    plt.subplot(2, 2, 2)
    plt.plot(training_history['train_losses'], label='训练损失')
    plt.plot(training_history['val_losses'], label='验证损失')
    plt.title('训练和验证损失')
    plt.xlabel('训练轮次')
    plt.ylabel('损失值')
    plt.legend()
    plt.grid(True)
    
    # 3. lambda_physics与损失比的关系
    plt.subplot(2, 2, 3)
    loss_ratio = [data_loss / (physics_loss + 1e-8) 
                  for data_loss, physics_loss in zip(training_history['data_losses'], training_history['physics_losses'])]
    plt.scatter(loss_ratio, training_history['lambda_history'], alpha=0.6)
    plt.title('λ_physics vs 损失比率')
    plt.xlabel('数据损失/物理损失比率')
    plt.ylabel('λ_physics')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 使用示例
if __name__ == "__main__":
    # 训练自适应物理感知模型
    trained_model, adaptive_criterion, history = train_adaptive_physics_modelall(train_loader,test_loader)
    print("自适应物理信息GRU模型训练完成！")
    print("最终超参数值:")
    print(f"λ_physics: {adaptive_criterion.lambda_physics.item():.6f}")
    print(f"扩散系数: {adaptive_criterion.diffusion_coef.item():.6f}")
    print(f"噪声强度: {adaptive_criterion.noise_intensity.item():.6f}")
    
    # 可视化超参数演化
    visualize_hyperparameter_evolution(history)

In [ ]:
import torch
import torch.nn as nn

# 设备配置
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# 使用正确的模型结构初始化
model = StochasticPhysicsInformedGRU(
    input_dim=1, 
    hidden_dim=128, 
    num_layers=2, 
    dropout=0.3,
    physics_dim=64
).to(device)

# 现在加载模型应该能正常工作
checkpoint = torch.load('best_adaptive_physics_modelall_fen.pth', map_location=device)

# 检查检查点结构
print("检查点包含的键:", checkpoint.keys())

# 加载模型状态字典
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    best_loss = checkpoint['best_loss']
    epoch = checkpoint.get('epoch', '未知')
else:
    # 如果检查点直接是状态字典
    model.load_state_dict(checkpoint)
    best_loss = '未知'
    epoch = '未知'

print(f"✅ 成功加载模型，验证损失: {best_loss:.6f}, 训练轮次: {epoch}")

# 初始化物理损失函数（使用与训练相同的参数）
criterion_physics = AdaptivePhysicsInformedLoss(
    alpha=0.6, 
    lambda_physics=0.2, 
    rho_air=1.225, 
    dt=1.0,
    diffusion_coef=0.01,
    noise_intensity=0.05
)

# 改进的评估函数：同时评估数据精度和物理一致性
def evaluate_model_with_physics(model, data_loader, physics_criterion, device, scaler=None):
    """
    综合考虑数据精度和物理一致性的模型评估函数
    
    Args:
        model: 训练好的模型
        data_loader: 测试数据加载器
        physics_criterion: 物理损失函数
        device: 计算设备
        scaler: 数据归一化器（可选，用于反归一化）
    
    Returns:
        avg_total_loss: 平均总损失
        avg_data_loss: 平均数据损失
        avg_physics_loss: 平均物理损失
        predictions: 预测值
        targets: 真实值
        physics_residuals: 物理残差统计
    """
    model.eval()
    total_total_loss = 0.0
    total_data_loss = 0.0
    total_physics_loss = 0.0
    all_predictions = []
    all_targets = []
    all_physics_residuals = []
    
    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)  # 确保形状匹配
            
            # 模型预测
            outputs, _ = model(batch_X)
            
            # 使用物理损失函数计算所有损失组件[1,4](@ref)
            total_loss_batch, data_loss_batch, physics_loss_batch = physics_criterion(
                outputs, batch_y, batch_X, device
            )
            
            # 累积损失
            total_total_loss += total_loss_batch.item()
            total_data_loss += data_loss_batch.item()
            total_physics_loss += physics_loss_batch.item()
            
            # 收集预测值和目标值
            all_predictions.extend(outputs.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())
            
            # 计算物理残差用于分析[2](@ref)
            residuals = physics_criterion._compute_stochastic_physics_constraint(
                outputs, batch_y, batch_X, device
            )
            all_physics_residuals.append(residuals.item())
    
    # 计算平均损失
    num_batches = len(data_loader)
    avg_total_loss = total_total_loss / num_batches
    avg_data_loss = total_data_loss / num_batches
    avg_physics_loss = total_physics_loss / num_batches
    
    # 转换为numpy数组
    predictions_array = np.array(all_predictions).flatten()
    targets_array = np.array(all_targets).flatten()
    physics_residuals_stats = {
        'mean': np.mean(all_physics_residuals),
        'std': np.std(all_physics_residuals),
        'min': np.min(all_physics_residuals),
        'max': np.max(all_physics_residuals)
    }
    
    return avg_total_loss, avg_data_loss, avg_physics_loss, predictions_array, targets_array, physics_residuals_stats

# ... 前面的代码保持不变 ...

# 进行物理一致性评估
avg_total_loss, avg_data_loss, avg_physics_loss, predictions, targets, physics_residuals = evaluate_model_with_physics(
    model, test_loader, criterion_physics, device, scaler
)

# 反归一化[5](@ref)
# 预测值包含三个分位数：[0.1, 0.5, 0.9]
predictions_original = scaler.inverse_transform(predictions.reshape(-1, 3))
predictions_0_1 = predictions_original[:, 0]
predictions_0_5 = predictions_original[:, 1]  # 0.5分位数作为点预测
predictions_0_9 = predictions_original[:, 2]

targets_original = scaler.inverse_transform(targets.reshape(-1, 1)).flatten()

# 计算点预测（0.5分位数）的标准评估指标
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0  # 避免除以零
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# 点预测指标
mse = mean_squared_error(targets_original, predictions_0_5)
rmse = np.sqrt(mse)
mae = mean_absolute_error(targets_original, predictions_0_5)
r2 = r2_score(targets_original, predictions_0_5)
mape = calculate_mape(targets_original, predictions_0_5)

# 计算点预测的标准差
point_prediction_errors = targets_original - predictions_0_5
point_prediction_sd = np.std(point_prediction_errors)

# 计算90%预测区间的评价指标
def calculate_interval_metrics(y_true, lower_bound, upper_bound):
    """
    计算区间预测的评价指标
    
    Args:
        y_true: 真实值
        lower_bound: 预测区间下界
        upper_bound: 预测区间上界
        
    Returns:
        coverage: 区间覆盖率
        avg_width: 平均区间宽度
        sharpness: 区间锐度（平均宽度与真实值范围的比值）
    """
    # 计算覆盖率
    covered = np.logical_and(y_true >= lower_bound, y_true <= upper_bound)
    coverage = np.mean(covered) * 100
    
    # 计算平均区间宽度
    avg_width = np.mean(upper_bound - lower_bound)
    
    # 计算区间锐度（平均宽度与真实值范围的比值）
    data_range = np.max(y_true) - np.min(y_true)
    sharpness = avg_width / data_range if data_range > 0 else 0
    
    return coverage, avg_width, sharpness

# 计算90%预测区间的指标
coverage, avg_width, sharpness = calculate_interval_metrics(
    targets_original, predictions_0_1, predictions_0_9
)

# 计算物理一致性指标
physics_consistency_ratio = avg_physics_loss / avg_data_loss if avg_data_loss > 0 else 0

print(f"\n=== 综合模型评估结果 ===")
print(f"\n📊 点预测（0.5分位数）指标:")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.4f}%")
print(f"标准差 (SD): {point_prediction_sd:.4f}")
print(f"R² Score: {r2:.4f}")

print(f"\n📊 区间预测（90%预测区间）指标:")
print(f"区间覆盖率: {coverage:.2f}%")
print(f"平均区间宽度: {avg_width:.4f}")
print(f"区间锐度: {sharpness:.4f}")

print(f"\n🔬 物理一致性指标:")
print(f"总损失: {avg_total_loss:.6f}")
print(f"数据损失: {avg_data_loss:.6f}")
print(f"物理损失: {avg_physics_loss:.6f}")
print(f"物理/数据损失比: {physics_consistency_ratio:.4f}")
print(f"物理残差均值: {physics_residuals['mean']:.6f}")
print(f"物理残差标准差: {physics_residuals['std']:.6f}")

print(f"\n📈 评估说明:")
if physics_consistency_ratio < 0.1:
    print("✅ 物理一致性良好：模型预测高度符合物理规律")
elif physics_consistency_ratio < 0.5:
    print("⚠️ 物理一致性中等：模型预测基本符合物理规律")
else:
    print("❌ 物理一致性较差：模型预测与物理规律偏差较大")

# 增强的可视化：包含物理一致性分析和预测区间
plt.figure(figsize=(16, 12))

# 1. 预测结果对比（含预测区间）
plt.subplot(2, 3, 1)
test_start_idx = train_size + window_size
test_indices = range(test_start_idx, test_start_idx + len(predictions_original))

# 绘制真实值
plt.plot(test_indices, targets_original, label='真实值', alpha=0.7, linewidth=2, color='blue')

# 绘制点预测（0.5分位数）
plt.plot(test_indices, predictions_0_5, label='预测值 (0.5分位)', alpha=0.7, linestyle='-', color='red')

# 绘制预测区间
plt.fill_between(test_indices, predictions_0_1, predictions_0_9, 
                color='orange', alpha=0.3, label='90%预测区间')

plt.title('时间序列预测结果对比')
plt.xlabel('时间步')
plt.ylabel('数值')
plt.legend()
plt.grid(True)

# 2. 散点图（预测vs真实）
plt.subplot(2, 3, 2)
plt.scatter(targets_original, predictions_0_5, alpha=0.5, c='green')
plt.plot([targets_original.min(), targets_original.max()], 
         [targets_original.min(), targets_original.max()], 'r--', linewidth=2)
plt.xlabel('真实值')
plt.ylabel('预测值 (0.5分位)')
plt.title('预测值 vs 真实值 (R² = {:.4f})'.format(r2))
plt.grid(True)

# 3. 残差分析
plt.subplot(2, 3, 3)
residuals = targets_original - predictions_0_5
plt.scatter(predictions_0_5, residuals, alpha=0.5, c='purple')
plt.axhline(y=0, color='red', linestyle='--', label='零误差线')
plt.axhline(y=residuals.mean(), color='blue', linestyle='-', label='残差均值')
plt.xlabel('预测值 (0.5分位)')
plt.ylabel('残差')
plt.title('预测残差分布 (标准差: {:.4f})'.format(point_prediction_sd))
plt.legend()
plt.grid(True)

# 4. 预测区间覆盖率可视化
plt.subplot(2, 3, 4)
# 随机选择100个点展示预测区间
sample_indices = np.random.choice(len(targets_original), min(100, len(targets_original)), replace=False)
sorted_indices = np.argsort(targets_original[sample_indices])

# 计算误差条（确保非负）
lower_errors = np.maximum(0, targets_original[sample_indices][sorted_indices] - predictions_0_1[sample_indices][sorted_indices])
upper_errors = np.maximum(0, predictions_0_9[sample_indices][sorted_indices] - targets_original[sample_indices][sorted_indices])

plt.errorbar(
    np.arange(len(sorted_indices)),
    targets_original[sample_indices][sorted_indices],
    yerr=[lower_errors, upper_errors],
    fmt='o', ecolor='lightblue', elinewidth=3, capsize=0,
    alpha=0.7, label='真实值±区间'
)

plt.plot(np.arange(len(sorted_indices)), predictions_0_5[sample_indices][sorted_indices], 
         'ro', alpha=0.5, label='预测值 (0.5分位)')
plt.title('预测区间覆盖情况 (覆盖率: {:.2f}%)'.format(coverage))
plt.xlabel('样本索引 (排序后)')
plt.ylabel('数值')
plt.legend()
plt.grid(True)

# 5. 物理残差分布
plt.subplot(2, 3, 5)
# 模拟物理残差数据（在实际中应从评估函数获取）
physics_residual_samples = np.random.normal(physics_residuals['mean'], 
                                          physics_residuals['std'], 
                                          1000)
plt.hist(physics_residual_samples, bins=50, alpha=0.7, color='orange', density=True)
plt.axvline(x=physics_residuals['mean'], color='red', linestyle='--', 
           label=f'均值: {physics_residuals["mean"]:.4f}')
plt.xlabel('物理残差值')
plt.ylabel('概率密度')
plt.title('物理约束残差分布')
plt.legend()
plt.grid(True)

# 6. 损失分量饼图
plt.subplot(2, 3, 6)
loss_components = [avg_data_loss, avg_physics_loss]
labels = ['数据损失', '物理损失']
colors = ['lightblue', 'lightcoral']
plt.pie(loss_components, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('测试损失分量构成')

plt.tight_layout()
plt.show()

# 物理一致性详细分析报告
print(f"\n=== 物理一致性详细分析 ===")
print(f"1. 随机扩散方程约束评估:")
print(f"   - 质量守恒残差: {physics_residuals['mean']:.6f}")
print(f"   - 动量守恒稳定性: {'良好' if physics_residuals['std'] < 0.1 else '需改进'}")
print(f"   - 随机项影响程度: {avg_physics_loss/avg_total_loss*100:.1f}%")

print(f"\n2. 模型物理合理性判断:")
if r2 > 0.9 and physics_consistency_ratio < 0.2:
    print("   ✅ 优秀：模型在数据精度和物理一致性方面均表现良好")
elif r2 > 0.7 and physics_consistency_ratio < 0.5:
    print("   ⚠️ 合格：模型基本满足要求，但仍有改进空间")
else:
    print("   ❌ 需改进：模型需要进一步优化物理约束")

print(f"\n3. 改进建议:")
if coverage < 85:
    print("   - 考虑调整分位数权重或模型结构以提高区间覆盖率")
if physics_consistency_ratio > 0.5:
    print("   - 增加物理损失权重 lambda_physics")
    print("   - 调整随机项参数（扩散系数、噪声强度）")
    print("   - 增加训练数据中的物理约束点")
elif r2 < 0.7:
    print("   - 优化模型架构（增加隐藏层维度）")
    print("   - 调整学习率和训练轮次")
    print("   - 检查数据质量和归一化处理")
else:
    print("   - 模型性能良好，可考虑部署应用")

In [ ]:

# 创建DataFrame
results_df = pd.DataFrame({
    'time_index': test_indices,
    'true_value': targets_original,
    'pred_low': predictions_0_1,
    'pred_med': predictions_0_5,
    'pred_high': predictions_0_9
})

# 保存为CSV文件
results_df.to_csv('quantile_prediction_results_'+'Aventa_'+'9_'+'all'+'.csv', index=False)
